In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [ ]:
ghg_capita_df = pd.read_csv('data/EDGAR_GHG_per_capita.csv')
gain_df = pd.read_csv('data/gain.csv')
gdb_df = pd.read_csv('data/gdp_input.csv')

## EDA vor merge der Daten

Bevor die drei Datensätze (GHG, ND-GAIN, GDP) zusammenführen, prüfen:
1. **Keys / Duplikate**: Eindeutigkeit der Identifikatoren
2. **Year-Coverage**: Zeitliche Abdeckung
3. **Missingness**: Fehlende Werte pro Land
4. **Einheiten und Skalen**: Bedeutung der Werte
5. **ISO3 Konsistenz**: Ländercode-Kompatibilität

In [ ]:
ghg_capita_df.head()

In [ ]:
gain_df.head()

In [ ]:
gdb_df.head()

## 1. Datenstruktur & Identifikatoren

In [ ]:
# Shape und Spalten der drei Datensätze
print("GHG per Capita (EDGAR)")
print(f"Shape: {ghg_capita_df.shape}")
print(f"Spalten: {list(ghg_capita_df.columns[:5])}... (erste 5)")
print(f"\nIdentifikator-Spalten:")
for col in ['EDGAR Country Code', 'Country']:
    if col in ghg_capita_df.columns:
        print(f"  - {col}: {ghg_capita_df[col].nunique()} unique values")

In [ ]:
print("ND-GAIN")
print(f"Shape: {gain_df.shape}")
print(f"Spalten: {list(gain_df.columns[:5])}... (erste 5)")
print(f"\nIdentifikator-Spalten:")
for col in ['ISO3', 'Name']:
    if col in gain_df.columns:
        print(f"  - {col}: {gain_df[col].nunique()} unique values")

In [ ]:
print("GDP")
print(f"Shape: {gdb_df.shape}")
print(f"Spalten: {list(gdb_df.columns[:5])}... (erste 5)")
print(f"\nIdentifikator-Spalten:")
for col in ['ISO3', 'Name']:
    if col in gdb_df.columns:
        print(f"  - {col}: {gdb_df[col].nunique()} unique values")

In [ ]:
# Prüfen auf Duplikate in den Identifier-Spalten
# GHG
ghg_dups = ghg_capita_df['EDGAR Country Code'].duplicated().sum()
print(f"GHG - Duplikate in 'EDGAR Country Code': {ghg_dups}")
if ghg_dups > 0:
    print(f"  Duplikate: {ghg_capita_df[ghg_capita_df['EDGAR Country Code'].duplicated(keep=False)][['EDGAR Country Code', 'Country']].head()}")

# ND-GAIN
gain_dups = gain_df['ISO3'].duplicated().sum()
print(f"ND-GAIN - Duplikate in 'ISO3': {gain_dups}")
if gain_dups > 0:
    print(f"  Duplikate: {gain_df[gain_df['ISO3'].duplicated(keep=False)][['ISO3', 'Name']].head()}")

# GDP
gdp_dups = gdb_df['ISO3'].duplicated().sum()
print(f"GDP - Duplikate in 'ISO3': {gdp_dups}")
if gdp_dups > 0:
    print(f"  Duplikate: {gdb_df[gdb_df['ISO3'].duplicated(keep=False)][['ISO3', 'Name']].head()}")

## 2. Zeitliche Abdeckung

In [ ]:
# Jahr-Spalten identifizieren (Spalten die wie "1995" aussehen)
def get_year_columns(df):
    """Findet alle Spalten die Jahr-Daten enthalten (z.B. '1995', '2000')"""
    year_cols = [c for c in df.columns if str(c).isdigit()]
    return sorted([int(y) for y in year_cols])

ghg_years = get_year_columns(ghg_capita_df)
gain_years = get_year_columns(gain_df)
gdp_years = get_year_columns(gdb_df)

# Überlappung
overlap = set(ghg_years) & set(gain_years) & set(gdp_years)
print(f"Überlappende Jahre (alle 3 Datensätze): {min(overlap) if overlap else 'N/A'} - {max(overlap) if overlap else 'N/A'}")
print(f"Anzahl überlappender Jahre: {len(overlap)}")
print(f"Jahre: {sorted(overlap)}")

## 3. ISO3-Code Konsistenz & Überlappung
Auch wenn es anders genannt wird, sind die EDGAR Country Codes vom ghg Datensatz auch ISO3-Codes.

In [ ]:
# ISO3-Code Überlappung zwischen den Datensätzen

ghg_codes = set(ghg_capita_df['EDGAR Country Code'].dropna().unique())
gain_codes = set(gain_df['ISO3'].dropna().unique())
gdp_codes = set(gdb_df['ISO3'].dropna().unique())

print(f"\nAnzahl Länder pro Datensatz:")
print(f"  GHG (EDGAR):  {len(ghg_codes)} Länder")
print(f"  ND-GAIN:      {len(gain_codes)} Länder")
print(f"  GDP:          {len(gdp_codes)} Länder")

# Überlappung
all_three = ghg_codes & gain_codes & gdp_codes
ghg_gain = ghg_codes & gain_codes
ghg_gdp = ghg_codes & gdp_codes
gain_gdp = gain_codes & gdp_codes

print(f"\nÜberlappung:")
print(f"  In ALLEN 3 Datensätzen:     {len(all_three)} Länder")
print(f"  In GHG + ND-GAIN:           {len(ghg_gain)} Länder")
print(f"  In GHG + GDP:               {len(ghg_gdp)} Länder")
print(f"  In ND-GAIN + GDP:           {len(gain_gdp)} Länder")

# Nur in einem Datensatz
only_ghg = ghg_codes - gain_codes - gdp_codes
only_gain = gain_codes - ghg_codes - gdp_codes
only_gdp = gdp_codes - ghg_codes - gain_codes
gain_not_ghg = gain_codes - ghg_codes

print(f"Nur in ND-GAIN:  {len(only_gain)} Länder")
print(f"Nur in GDP:      {len(only_gdp)} Länder")
print(f"Nur in GHG:      {len(only_ghg)} Länder")
print(only_ghg)
print(f"In ND-GAIN/GDP, aber nicht GHG:  {len(gain_not_ghg)} Länder")
print(gain_not_ghg)
    


Im ghg_capita sind gewisse Länder zusammengenommen. z.B. MNE und SRB sind in GHG als SCG zusammengenommen oder LIE gehört dort zu CHE. -> Allenfalls in ND-Gain und GDP Datensätzen diese Zeilen zusammenaddieren? (Achtung, so wird wahrscheinlich ND-Gain verzogen?)

In ghg_capita ausserdem noch EU27 und Global Total vor mergen entfernen.

## 4. Missingness (Fehlende Werte)

In [ ]:
# GHG
print("\nGHG per Capita (EDGAR):")
print(f"{ghg_capita_df.isnull().sum().sum()} fehlende Werte gefunden")

# ND-GAIN
print("\nND-GAIN:")
print(f"{gain_df.isnull().sum().sum()} fehlende Werte gefunden")

# GDP
print("\nGDP:")
print(f"{gdb_df.isnull().sum().sum()} fehlende Werte gefunden")